# Navigation dynamics

> An exact MultiGrid navigation model satisfying the `Dynamics` protocol, to plan without a learned model.

In [ ]:
#| default_exp envs.multigrid.dynamics

In [ ]:
#| hide
from nbdev.showdoc import *

`NavigationDynamics` implements the MultiGrid movement rules in torch, over the symbolic
infos: actions `left` / `right` turn the agent, `forward` moves it one cell unless the cell is
blocked (walls, closed doors, objects); other actions do nothing. Its embedding is the agents'
positions, so with `GoalMSE` the planning cost is the squared distance to ``goal_position``.

It is the known-dynamics baseline of model-based planning: it exercises the whole pipeline
(evaluator, objective, solver, `WorldModelPolicy`) without training anything, and learned world
models take its place. It ignores other agents (they never block each other) and picking up or
toggling objects.

In [ ]:
#| export
from __future__ import annotations

import torch

from stable_marl.envs.multigrid.core.actions import Action
from stable_marl.envs.multigrid.core.constants import State, Type

In [ ]:
#| export
_DIR_TO_VEC = torch.tensor([[1, 0], [0, 1], [-1, 0], [0, -1]])           # right, down, left, up
_BLOCKING = torch.tensor([Type.wall.to_index(), Type.box.to_index(), Type.ball.to_index(), Type.key.to_index()])


def walkable(state: torch.Tensor) -> torch.Tensor:
    "Cells an agent can move to, from the symbolic state (..., W, H, 3): (..., W, H) bool."
    obj_type, obj_state = state[..., 0], state[..., 2]
    closed_door = (obj_type == Type.door.to_index()) & (obj_state != State.open.to_index())
    return ~(torch.isin(obj_type, _BLOCKING.to(obj_type.device)) | closed_door)


class NavigationDynamics(torch.nn.Module):
    """
    Exact MultiGrid navigation model (``Dynamics`` protocol). Reads the infos ``position``
    (..., num_agents, 2), ``direction`` (..., num_agents) and ``state`` (..., W, H, 3), at the
    last context frame (it is Markovian: earlier frames and ``action_history`` are not needed).
    Each planning step covers `action_block` env steps.
    """
    def __init__(self, action_block: int = 1):
        super().__init__()
        self.action_block = action_block

    def encode(self, x: dict) -> dict:
        "Embedding: the agents' positions, ``emb`` (..., num_agents, 2)."
        x['emb'] = x['position'].float()
        return x

    def rollout(self, info_dict: dict, action_candidates: torch.Tensor) -> dict:
        """
        Roll the candidates out from the last context frame: one-hot ``(B, S, horizon, num_agents,
        action_block * K)``, or action indices ``(B, S, horizon, num_agents[, action_block])``. Adds
        ``predicted_emb`` ``(B, S, 1 + horizon, num_agents, 2)``: the positions after each planning
        step, the current one first.
        """
        ab = self.action_block
        if action_candidates.is_floating_point():
            actions = action_candidates.reshape(*action_candidates.shape[:-1], ab, -1).argmax(-1)
        else:
            actions = action_candidates if action_candidates.ndim == 5 else action_candidates.unsqueeze(-1)
        pos = info_dict['position'][:, :, -1].long()                     # (B, S, A, 2)
        direction = info_dict['direction'][:, :, -1].long()               # (B, S, A)
        free = walkable(info_dict['state'][:, :, -1])                     # (B, S, W, H)
        B, S, A, _ = pos.shape
        W, H = free.shape[-2:]
        b = torch.arange(B, device=pos.device)[:, None, None].expand(B, S, A)
        s = torch.arange(S, device=pos.device)[None, :, None].expand(B, S, A)
        dir_to_vec = _DIR_TO_VEC.to(pos.device)
        trajectory = [pos]
        for t in range(actions.shape[2]):
            for j in range(actions.shape[4]):
                a = actions[:, :, t, :, j]
                direction = torch.where(a == int(Action.left), (direction - 1) % 4,
                                        torch.where(a == int(Action.right), (direction + 1) % 4, direction))
                target = pos + dir_to_vec[direction]
                inside = (target[..., 0] >= 0) & (target[..., 0] < W) & (target[..., 1] >= 0) & (target[..., 1] < H)
                tx, ty = target[..., 0].clamp(0, W - 1), target[..., 1].clamp(0, H - 1)
                move = (a == int(Action.forward)) & inside & free[b, s, tx, ty]
                pos = torch.where(move[..., None], target, pos)
            trajectory.append(pos)
        info_dict['predicted_emb'] = torch.stack(trajectory, dim=2).float()
        return info_dict

### Tests

In [ ]:
import numpy as np
from stable_marl.envs import make
from stable_marl.protocols import Dynamics
from stable_marl.wrappers.default import MegaWrapper

assert isinstance(NavigationDynamics(), Dynamics)
# the model predicts the env's positions exactly (single agent, so no agent-agent blocking)
env = MegaWrapper(make('MultiGrid-FindGoal-15x15-v0', agents=1, num_obstacles=8))
_, info = env.reset(seed=0)
rng = np.random.default_rng(0)
actions = rng.integers(3, size=40)
as_tensor = lambda v: torch.as_tensor(np.asarray(v))[None, None, None]       # (B=1, S=1, T=1, ...)
ctx = {k: as_tensor(info[k]) for k in ('position', 'direction', 'state')}
predicted = NavigationDynamics().rollout(ctx, torch.as_tensor(actions)[None, None, :, None])['predicted_emb'][0, 0]
for t, a in enumerate(actions):
    _, _, term, trunc, info = env.step({0: int(a)})
    assert predicted[t + 1, 0].tolist() == info['position'][0].tolist(), t
    if term[0] or trunc[0]: break

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()